# Myllia | Echoes of Silenced Genes — Clean Notebook

Minimal end-to-end pipeline (Public 60 only):
1) Load `training_cells.h5ad` (single-cell)
2) Build perturbation-level features via batch-aware logFC vs `non-targeting`
3) Train output-PCA + Ridge
4) Write `submission.csv` (Public overwritten, Private kept as baseline)


## Config


In [1]:
import os

# Set this to your Kaggle input directory
DATA_DIR = os.environ.get('DATA_DIR', '/kaggle/input/datasets/jinsei0837/myllia-echoes-of-silenced-genes-a-cell-challenge')

# AnnData obs columns
PERT_COL = 'sgrna_symbol'
BATCH_COL = 'channel'
CONTROL_LABEL = 'non-targeting'

print('DATA_DIR:', DATA_DIR)


DATA_DIR: /kaggle/input/datasets/jinsei0837/myllia-echoes-of-silenced-genes-a-cell-challenge


## Pipeline


In [2]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/datasets/jinsei0837/myllia-echoes-of-silenced-genes-a-cell-challenge/sample_submission.csv
/kaggle/input/datasets/jinsei0837/myllia-echoes-of-silenced-genes-a-cell-challenge/training_data_ground_truth_table.csv
/kaggle/input/datasets/jinsei0837/myllia-echoes-of-silenced-genes-a-cell-challenge/training_data_means.csv
/kaggle/input/datasets/jinsei0837/myllia-echoes-of-silenced-genes-a-cell-challenge/pert_ids_val.csv
/kaggle/input/datasets/jinsei0837/myllia-echoes-of-silenced-genes-a-cell-challenge/training_cells.h5ad


In [3]:
df_means = pd.read_csv(f"{DATA_DIR}/training_data_means.csv")
df_gt    = pd.read_csv(f"{DATA_DIR}/training_data_ground_truth_table.csv")

# 特徴エンジニアリング

In [4]:
!pip install -q scanpy anndata


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.1/2.1 MB 25.9 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 176.6/176.6 kB 15.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 58.6/58.6 kB 4.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 284.1/284.1 kB 22.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.2/9.2 MB 106.2 MB/s eta 0:00:0000:0100:01


In [5]:
#h5daを読む
import scanpy as sc

adata = sc.read_h5ad(f"{DATA_DIR}/training_cells.h5ad")
adata

AnnData object with n_obs × n_vars = 17882 × 19226
    obs: 'nCount_RNA', 'nFeature_RNA', 'percent.mt', 'sgrna_id', 'sgrna_symbol', 'channel'
    var: 'features'

In [6]:
adata.obs.columns

Index(['nCount_RNA', 'nFeature_RNA', 'percent.mt', 'sgrna_id', 'sgrna_symbol',
       'channel'],
      dtype='object')

コントロールラベルは？

In [7]:
# perturbationの候補（上位を表示）
adata.obs["sgrna_symbol"].value_counts().head(20)

sgrna_symbol
non-targeting    1026
MAPK1             447
KDM5C             428
FLNA              415
HDAC4             410
HSPA4             382
RUNX1             374
PAGR1             355
PAXIP1            350
INSIG1            348
FGFR1             312
KDM4A             311
HSPD1             309
METTL3            309
CHMP4B            299
HDAC8             296
IKBKG             294
DZIP3             288
KAT2A             288
APAF1             279
Name: count, dtype: int64

non-targetingがコントロール。細胞数も十分。

In [8]:
# channelの中身（バッチっぽいので一応見る）
adata.obs["channel"].value_counts()

channel
ch_3    4586
ch_1    4528
ch_4    4397
ch_2    4371
Name: count, dtype: int64

In [9]:
#h5ad から X_sc（perturbation×遺伝子特徴）を作る
gene_names = adata.var_names

def mean_vec(mat):
    m = mat.mean(axis=0)
    return np.asarray(m).ravel()

# バッチごとの control 平均
ctrl_means = {}
for ch in adata.obs[BATCH_COL].unique():
    m = (adata.obs[BATCH_COL].eq(ch) & adata.obs[PERT_COL].eq(CONTROL_LABEL))
    if m.sum() == 0:
        continue
    ctrl_means[ch] = mean_vec(adata[m].X)

# perturbation ごとの logFC（バッチ内差分→バッチ平均）
features = {}
for pert in adata.obs[PERT_COL].unique():
    if pert == CONTROL_LABEL:
        continue

    per_ch = []
    for ch, ctrl_mean in ctrl_means.items():
        m = (adata.obs[BATCH_COL].eq(ch) & adata.obs[PERT_COL].eq(pert))
        if m.sum() == 0:
            continue
        pert_mean = mean_vec(adata[m].X)
        logfc = np.log1p(pert_mean) - np.log1p(ctrl_mean)
        per_ch.append(logfc)

    if len(per_ch) == 0:
        continue

    features[pert] = np.mean(per_ch, axis=0)

X_sc = pd.DataFrame.from_dict(features, orient="index", columns=gene_names)
print("X_sc:", X_sc.shape)


X_sc: (80, 19226)


In [10]:
#学習用 y を「submissionに必要な遺伝子だけ」に揃える
df_gt = pd.read_csv(f"{DATA_DIR}/training_data_ground_truth_table.csv").set_index("pert_id")
sample = pd.read_csv(f"{DATA_DIR}/sample_submission.csv")

target_cols = [c for c in sample.columns if c != "pert_id"]  # 5127 genes

# y: trainで使えるpertのみ、かつ提出対象遺伝子に限定
y_train = df_gt.loc[df_gt.index.intersection(X_sc.index), target_cols].copy()
X_train = X_sc.loc[y_train.index].copy()

print("X_train:", X_train.shape, "y_train:", y_train.shape)


X_train: (80, 19226) y_train: (80, 5127)


In [11]:
#PCA（出力側）+ Ridge で学習 → val 予測
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.linear_model import Ridge

# スケール
scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)

# 出力PCA
pca_y = PCA(n_components=10, random_state=42)
y_train_pca = pca_y.fit_transform(y_train)

# Ridge
model = Ridge(alpha=300, random_state=42)
model.fit(X_train_s, y_train_pca)


Ridge(alpha=300, random_state=42)

In [12]:
pert_val = pd.read_csv(f"{DATA_DIR}/pert_ids_val.csv")  # columns: pert, class, pert_id

# 提出遺伝子を X_sc から取る（無ければ平均で埋める）
X_mean = X_train.mean(axis=0)

X_val_list = []
for g in pert_val["pert"]:
    if g in X_sc.index:
        X_val_list.append(X_sc.loc[g].values)
    else:
        X_val_list.append(X_mean.values)  # fallback

X_val = pd.DataFrame(
    np.vstack(X_val_list),
    columns=X_train.columns
)

X_val_s = scaler.transform(X_val)

y_val_pca = model.predict(X_val_s)
y_val_pred = pca_y.inverse_transform(y_val_pca)  # (n_val, 5127)


In [13]:
# --- submission（120行）を土台にして、public 60だけ上書きする ---
sample = pd.read_csv(f"{DATA_DIR}/sample_submission.csv")

target_cols = [c for c in sample.columns if c != "pert_id"]  # 5127 genes

# 予測（60行）を pert_id 付き DataFrame にする
pred_df = pd.DataFrame(y_val_pred, columns=target_cols)
pred_df.insert(0, "pert_id", pert_val["pert_id"].values)

# sample（120行）に left merge して、predがあるところだけ上書き
sub = sample.merge(pred_df, on="pert_id", how="left", suffixes=("", "_pred"))

for c in target_cols:
    sub[c] = sub[f"{c}_pred"].combine_first(sub[c])  # predがある行だけ上書き
    sub.drop(columns=[f"{c}_pred"], inplace=True)

# 欠損チェック（0になるのが正解）
print("NaN rows:", sub.isna().any(axis=1).sum())
print("NaN cells:", int(sub.isna().sum().sum()))

sub.to_csv("submission.csv", index=False)
print("saved: submission.csv", sub.shape)


NaN rows: 0
NaN cells: 0
saved: submission.csv (120, 5128)
